# Implementation of Multi-Format CLR

In [2]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # disable GPU devices
os.environ["TFDS_DATA_DIR"] = os.path.expanduser("~/tensorflow_datasets")  # default location of tfds database
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers, models, regularizers
from keras.utils import plot_model
from keras.callbacks import ReduceLROnPlateau, EarlyStopping

import tensorflow as tf
import tensorflow_datasets as tfds

from IPython.display import Image

import librosa.display

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.manifold import TSNE

import numpy as np

# Turn off logging for TF
import logging
logging.disable(logging.WARNING)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"
tf.get_logger().setLevel(logging.ERROR)

from dpmhm.datasets import preprocessing, feature, utils, transformer, _DTYPE, _ENCLEN

ds_all, ds_info = tfds.load(
    'CWRU',
    with_info=True,
)

ds0 = ds_all['train']
ds0.element_spec

2024-07-10 09:09:38.926374: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-07-10 09:09:38.932046: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-07-10 09:09:39.020952: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-07-10 09:09:40.837848: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2024-07-10 09:09:45.480259: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:282] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2024-07-10 09:09:45.480330: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:134] ret

{'metadata': {'Dataset': TensorSpec(shape=(), dtype=tf.string, name=None),
  'FaultComponent': TensorSpec(shape=(), dtype=tf.string, name=None),
  'FaultLocation': TensorSpec(shape=(), dtype=tf.string, name=None),
  'FaultSize': TensorSpec(shape=(), dtype=tf.float32, name=None),
  'FileName': TensorSpec(shape=(), dtype=tf.string, name=None),
  'LoadForce': TensorSpec(shape=(), dtype=tf.uint32, name=None),
  'NominalRPM': TensorSpec(shape=(), dtype=tf.uint32, name=None),
  'RPM': TensorSpec(shape=(), dtype=tf.uint32, name=None)},
 'sampling_rate': TensorSpec(shape=(), dtype=tf.uint32, name=None),
 'signal': {'BA': TensorSpec(shape=(None,), dtype=tf.float32, name=None),
  'DE': TensorSpec(shape=(None,), dtype=tf.float32, name=None),
  'FE': TensorSpec(shape=(None,), dtype=tf.float32, name=None)}}

Parameters

In [3]:
batch_size = 32
n_embedding  = 128 
spectrogram_kernel_size = (3,3) 
audio_kernel_size=(3)
tau = 0.1
projection_dim = 128 
img_dim =64
seg_length = 6000
nb_elements_per_second=1

# Preprocessing on data

In [4]:
compactor = transformer.DatasetCompactor(ds0,
                                         channels=['DE', 'FE', 'BA'],
                                         keys=['FaultLocation', 'FaultComponent', 'FaultSize'],
                                         resampling_rate=12000,window_size = int(12000), hop_size=int(12000/2))

def spectrogram(signal, sampling_rate, extractor: callable) -> dict:
    """Feature transform of a signal element.

    The transformed element has a dictionary structure which contains
    """
    Xf = tf.py_function(
        func=lambda x, sr: extractor(x.numpy(), sr),  
        inp=[signal, sampling_rate],
        Tout=_DTYPE
    )
    # Xf.set_shape((signal.shape[0], None, None))
    Xf = tf.expand_dims(Xf, axis=-1)  # Add a channel dimension
    Xf = tf.image.resize(Xf, (64, 64), method=tf.image.ResizeMethod.BICUBIC)
    Xf = tf.squeeze(Xf, axis=-1)  # Remove the channel dimension
    return Xf

_func = lambda x, sr: feature.spectral_features(x, sr, 'spectrogram', 
                                                time_window=0.025, hop_step=0.0125, normalize=False,
                                                to_db=True)[0]

def create_pairs(element, seg_length, nb_elements_per_second):
    signal = element['signal']
    label = element['label']
    sampling_rate = element['sampling_rate']

    pairs = []

    for _ in range(nb_elements_per_second):
        start_idx_1 = tf.random.uniform(shape=(), minval=0, maxval=signal.shape[1] - seg_length, dtype=tf.int32)
        start_idx_2 = tf.random.uniform(shape=(), minval=0, maxval=signal.shape[1] - seg_length, dtype=tf.int32)

        # Extraire des segments de chaque canal
        sig1 = signal[:, start_idx_1:start_idx_1 + seg_length]
        sig2 = signal[:, start_idx_2:start_idx_2 + seg_length]

        # Calculer les caractéristiques spectrogramme pour le deuxième signal
        sig2_features = spectrogram(sig2, sampling_rate, _func)

        pairs.append(((sig1, label), (sig2_features, label)))

    return pairs

def flatten_pairs(element, seg_length, nb_elements_per_second):
    pairs = create_pairs(element, seg_length, nb_elements_per_second)
    sig1_list = []
    sig2_features_list = []
    label_list = []

    for pair in pairs:
        sig1_list.append(pair[0][0])
        sig2_features_list.append(pair[1][0])
        label_list.append(pair[0][1])
    
    return (sig1_list, sig2_features_list, label_list)

def generator(seg_length, nb_elements_per_second):
    for element in compactor.dataset:
        sig1_list, sig2_features_list, label_list = flatten_pairs(element, seg_length, nb_elements_per_second)
        for sig1, sig2_features, label in zip(sig1_list, sig2_features_list, label_list):
            yield (sig1, label), (sig2_features, label)

# contains element under the form ((audio, label),(spectrogram, label))
ds = tf.data.Dataset.from_generator(lambda: generator(seg_length, nb_elements_per_second), output_signature=(
    ((tf.TensorSpec(shape=(3, seg_length), dtype=tf.float32), tf.TensorSpec(shape=(), dtype=tf.string)),
    (tf.TensorSpec(shape=(3,img_dim, img_dim), dtype=tf.float32), tf.TensorSpec(shape=(), dtype=tf.string)))))

def preproc(x, y):
    label_layer = layers.StringLookup(
        # num_oov_indices=0,   # force zero-based integer
        vocabulary=labels,
        # output_mode='one_hot'
    )
    return ((tf.transpose(x[0], [1, 0]), tf.cast(label_layer(x[1]), tf.int32)),
            (tf.transpose(y[0], [1, 2, 0]), tf.cast(label_layer(x[1]), tf.int32)))

labels = list(compactor.full_label_dict.keys())
ds_window = ds.map(lambda x, y : preproc(x,y), num_parallel_calls=tf.data.AUTOTUNE)

ds_size = sum([1 for _ in ds])
img_input_shape=(img_dim,img_dim, 3)
audio_input_shape=(seg_length, 3)
splits = {'train':0.7, 'val':0.2, 'test':0.1}
ds_split = utils.split_dataset(ds_window, splits, ds_size=int(ds.cardinality()))

2024-07-10 09:09:49.340536: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-10 09:09:50.487846: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-10 09:09:51.310276: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-10 09:09:52.847589: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-10 09:10:20.724780: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-10 09:10:34.285580: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-10 09:11:14.892811: W tensorflow/core/framework/local_rendezvous.cc:404] L

Create the encoders

In [5]:
@keras.utils.register_keras_serializable()
class SpectrogramEncoder(models.Model):
    def __init__(self, input_shape, n_embedding,kernel_size):
        self.input_shape = input_shape
        activation = 'relu'
        padding = 'same'
        strides = (2,2)
        pool_size = (2,2)
        a_reg = 0. 

        super(SpectrogramEncoder, self).__init__()

        # Use more blocks and larger kernel size to get more smoothing in the reconstruction.
        input_layer= layers.Input(shape=input_shape, name='input_enc_spectrogram')

        layers_encoder = [
            # Block 1
            layers.Conv2D(32, kernel_size=kernel_size, activation=activation, padding=padding, name='conv1_enc'),
            layers.MaxPooling2D(pool_size=pool_size, strides=strides, name='pool1_enc'),
            layers.BatchNormalization(name='bn1_enc'), # by default axis=-1 for channel-last

            # Block 2
            layers.Conv2D(64, kernel_size=kernel_size, activation=activation, padding=padding, name='conv2_enc'),
            layers.MaxPooling2D(pool_size=pool_size, strides=strides, name='pool2_enc'),
            layers.BatchNormalization(name='bn2_enc'),

            # Block 3
            layers.Conv2D(128, kernel_size=kernel_size, activation=activation, padding=padding, name='conv3_enc'),
            layers.MaxPooling2D(pool_size=pool_size, strides=strides, name='pool3_enc'),
            layers.BatchNormalization(name='bn3_enc'),

            # Block fc
            layers.Flatten(name='flatten'),
            layers.Dense(n_embedding, activation=activation,activity_regularizer=regularizers.L1(a_reg), name='fc1_enc') if a_reg > 0
            else layers.Dense(n_embedding, activation=activation, name='fc1_enc')
        ]

        self.encoder = models.Sequential([input_layer] +layers_encoder, name='spectrogram_encoder')

    def call(self, x):
        return self.encoder(x)
    
spectrogram_encoder = SpectrogramEncoder(img_input_shape,n_embedding,spectrogram_kernel_size)

In [6]:
@keras.utils.register_keras_serializable()
class audioEncoder(models.Model):
    def __init__(self, input_shape, n_embedding, kernel_size):
        self.input_shape = input_shape
        activation = 'relu'
        padding = 'same'
        strides = 2
        pool_size = 2
        a_reg = 0.

        super(audioEncoder, self).__init__()

        input_layer = layers.Input(shape=input_shape, name='input_enc_audio')

        layers_encoder = [
            # Block 1
            layers.Conv1D(64, kernel_size=kernel_size, activation=activation, padding=padding, name='conv1_enc'),
            layers.MaxPooling1D(pool_size=pool_size, strides=strides, name='pool1_enc'),
            layers.BatchNormalization(name='bn1_enc'),

            # Block 2
            layers.Conv1D(128, kernel_size=kernel_size, activation=activation, padding=padding, name='conv2_enc'),
            layers.MaxPooling1D(pool_size=pool_size, strides=strides, name='pool2_enc'),
            layers.BatchNormalization(name='bn2_enc'),

            # Block 3
            layers.Conv1D(256, kernel_size=kernel_size, activation=activation, padding=padding, name='conv3_enc'),
            layers.MaxPooling1D(pool_size=pool_size, strides=strides, name='pool3_enc'),
            layers.BatchNormalization(name='bn3_enc'),

            # Block 4
            layers.Conv1D(512, kernel_size=kernel_size, activation=activation, padding=padding, name='conv4_enc'),
            layers.MaxPooling1D(pool_size=pool_size, strides=strides, name='pool4_enc'),
            layers.BatchNormalization(name='bn4_enc'),

            # Block 5
            layers.Conv1D(1024, kernel_size=kernel_size, activation=activation, padding=padding, name='conv5_enc'),
            layers.MaxPooling1D(pool_size=pool_size, strides=strides, name='pool5_enc'),
            layers.BatchNormalization(name='bn5_enc'),

            # Block fc
            layers.Flatten(name='flatten'),
            layers.Dense(n_embedding, activation=activation, activity_regularizer=regularizers.L1(a_reg), name='fc1_enc') if a_reg > 0
            else layers.Dense(n_embedding, activation=activation, name='fc1_enc')
        ]

        self.encoder = models.Sequential([input_layer] + layers_encoder, name='audio_encoder')

    def call(self, x):
        return self.encoder(x)

audio_encoder = audioEncoder(audio_input_shape, n_embedding, audio_kernel_size)

**Contrastive learning on the encoder**

In [7]:
# Prepare the training data
ds_train = ds_split['train'].shuffle(ds_size, reshuffle_each_iteration=True).cache().batch(batch_size,drop_remainder=True).prefetch(tf.data.AUTOTUNE)
ds_val = ds_split['val'].batch(batch_size,drop_remainder=True)
ds_test = ds_split['test'].batch(1)

Contrastive loss function

In [8]:
def contrastive_loss_fn(z_i, z_j, tau=0.5):
    z_i = tf.math.l2_normalize(z_i, axis=1)
    z_j = tf.math.l2_normalize(z_j, axis=1)

    # Compute the similarity matrix
    similarity_matrix = tf.matmul(z_i, z_j, transpose_b=True) / tau

    # Compute the positive similarity
    positive_similarity = tf.linalg.diag_part(similarity_matrix)

    # Compute the negative similarity
    negative_similarity = tf.linalg.set_diag(similarity_matrix, tf.zeros_like(tf.linalg.diag_part(similarity_matrix)))

    # Compute the numerator of the loss function
    numerator = tf.exp(positive_similarity)

    # Compute the denominator of the loss function
    denominator = tf.reduce_sum(tf.exp(negative_similarity), axis=1)

    # Compute the loss function
    loss = -tf.reduce_mean(tf.math.log(numerator / denominator))

    return loss

Audio augmentation

In [9]:
class SingleaudioAugmenter(keras.layers.Layer):
    '''
    Apply the following data augment to the audio signal:
    1. Mixes with a Gaussian noise of small amplitude
    2. Randomly masks a small interval of time
    3. Randomly masks a small interval of frequency
    4. Applys a frequency shift
    '''
    def __init__(self, noise_amplitude=0.1, **kwargs):
        super(SingleaudioAugmenter, self).__init__(**kwargs)
        self.noise_amplitude = noise_amplitude

    def call(self, inputs):
        gaussian_noise = tf.keras.backend.random_normal(inputs.shape, mean=0, stddev=self.noise_amplitude)
        alpha = tf.random.uniform(shape=[], minval=0, maxval=1, dtype=tf.float32)
        mixed_audio = alpha * inputs + (1 - alpha) * gaussian_noise

        t = tf.minimum(tf.random.uniform(shape=[], minval=inputs.shape[0] // 50, maxval=inputs.shape[0] // 20, dtype=tf.int32), inputs.shape[0])
        t0 = tf.random.uniform(shape=[], minval=0, maxval=inputs.shape[0] - t, dtype=tf.int32)

        mask = tf.ones_like(inputs)
        mask = tf.tensor_scatter_nd_update(mask, tf.range(t0, t0 + t)[:, None], tf.zeros((t,), dtype=inputs.dtype))
        masked_audio = mixed_audio * mask

        spectrogram = tf.abs(tf.signal.stft(masked_audio, frame_length=256, frame_step=128, fft_length=256))

        f = tf.random.uniform(shape=[], minval=1, maxval=spectrogram.shape[1] // 4, dtype=tf.int32)
        masked_spectrogram = tf.tensor_scatter_nd_update(spectrogram, tf.range(f)[:, None], tf.zeros((f, spectrogram.shape[1]), dtype=spectrogram.dtype))

        shift_freq = tf.random.uniform(shape=[], minval=-2, maxval=2, dtype=tf.float32)
        shifted_spectrogram = self.frequency_shift(masked_spectrogram, shift_freq, 12000)

        augmented_audio = tf.expand_dims(shifted_spectrogram, axis=-1)
        augmented_audio = tf.image.resize_with_crop_or_pad(augmented_audio, inputs.shape[0], 1)
        augmented_audio = tf.squeeze(augmented_audio, axis=-1)
        augmented_audio = tf.cast(augmented_audio, tf.float32)
        return augmented_audio

    def frequency_shift(self, spectrogram, shift_freq, fs):
        freq_bins, time_bins = tf.shape(spectrogram)[0], tf.shape(spectrogram)[1]
        X = tf.cast(tf.signal.fft(tf.cast(spectrogram, tf.complex64)), tf.complex64)
        phase_shift_real = tf.cos(-2 * np.pi * shift_freq * tf.cast(tf.range(freq_bins), tf.float32) / fs)
        phase_shift_imag = tf.sin(-2 * np.pi * shift_freq * tf.cast(tf.range(freq_bins), tf.float32) / fs)
        phase_shift = tf.complex(phase_shift_real, phase_shift_imag)
        phase_shift = tf.reshape(phase_shift, (freq_bins, 1)) 
        X_shifted = X * phase_shift
        return tf.signal.ifft(X_shifted)

class audioAugmenter(keras.layers.Layer):
    def __init__(self, audio_augmenter, **kwargs):
        super(audioAugmenter, self).__init__(**kwargs)
        self.audio_augmenter = audio_augmenter

    def call(self, inputs):
        augmented_signals = []
        for signal in inputs:
            signal_augmented = []
            for elem in signal:
                augmented_signal = self.audio_augmenter(elem)
                signal_augmented.append(augmented_signal)
            signal_augmented = tf.stack(signal_augmented, axis=0)
            augmented_signals.append(signal_augmented)
        augmented_signals = tf.stack(augmented_signals, axis=0)
        return augmented_signals

Create the MF-CLR model

In [10]:
tf.config.experimental_run_functions_eagerly(True)

# Define the contrastive model with model-subclassing
class MF_CLRModel(keras.Model):
    def __init__(self):
        super().__init__()

        self.tau = tau

        self.spectrogram_augmenter = keras.Sequential([
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomZoom(0.2),
            layers.RandomTranslation(height_factor=0.2, width_factor=0.2),
        ], name='Spectrogram_augmenter')

        self.audio_augmenter = audioAugmenter(SingleaudioAugmenter())
        
        self.spectrogram_encoder = spectrogram_encoder.encoder

        self.audio_encoder = audio_encoder.encoder

        self.spectrogram_projection_head = keras.Sequential([
                layers.Dense(256, activation='relu'),
                layers.BatchNormalization(),
                layers.Dense(128, activation='relu'),
                layers.BatchNormalization(),
                layers.Dense(projection_dim),
            ], name='Spectrogram_projection_head')
        
        self.audio_projection_head = keras.Sequential([
                layers.Dense(256, activation='relu'),
                layers.BatchNormalization(),
                layers.Dense(128, activation='relu'),
                layers.BatchNormalization(),
                layers.Dense(projection_dim),
            ], name='audio_Projection_head')

    def compile(self, audio_optimizer, spectrogram_optimizer,**kwargs):
        super().compile(**kwargs)

        self.audio_optimizer = audio_optimizer
        self.spectrogram_optimizer = spectrogram_optimizer

        self.contrastive_loss_tracker = keras.metrics.Mean(name="c_loss")
        self.mean_cosine_similarity = keras.metrics.Mean(name="mean_cosine_similarity")

    @property
    def metrics(self):
        return [
            self.contrastive_loss_tracker,
            self.mean_cosine_similarity,
        ]

    def train_step(self, data):
        ((audio, _), (spectrogram, _)) = data
        augmented_audio=audio
        # augmented_audio = self.audio_augmenter(audio)
        augmented_spectrogram = self.spectrogram_augmenter(spectrogram)

        with tf.GradientTape() as tape_audio, tf.GradientTape() as tape_spectrogram:
            features_audio = self.audio_encoder(augmented_audio)
            projections_audio = self.audio_projection_head(features_audio)

            features_spectrogram = self.spectrogram_encoder(augmented_spectrogram)

            projections_spectrogram = self.spectrogram_projection_head(features_spectrogram)

            contrastive_loss = contrastive_loss_fn(projections_audio, projections_spectrogram, self.tau) +contrastive_loss_fn(projections_spectrogram, projections_audio, self.tau)

        gradients_audio = tape_audio.gradient(
            contrastive_loss,
            self.audio_encoder.trainable_weights + self.audio_projection_head.trainable_weights,
        )
        self.audio_optimizer.apply_gradients(
            zip(gradients_audio, self.audio_encoder.trainable_weights + self.audio_projection_head.trainable_weights)
        )

        gradients_spectrogram = tape_spectrogram.gradient(
            contrastive_loss,
            self.spectrogram_encoder.trainable_weights + self.spectrogram_projection_head.trainable_weights,
        )
        self.spectrogram_optimizer.apply_gradients(
            zip(gradients_spectrogram, self.spectrogram_encoder.trainable_weights + self.spectrogram_projection_head.trainable_weights)
        )

        self.contrastive_loss_tracker.update_state(contrastive_loss)
        cosine_similarity = tf.reduce_mean(keras.losses.cosine_similarity(projections_audio, projections_spectrogram))
        self.mean_cosine_similarity.update_state(cosine_similarity)

        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        ((audio, _), (spectrogram, _)) = data
        augmented_audio=audio
        # augmented_audio = self.audio_augmenter(audio)
        augmented_spectrogram = self.spectrogram_augmenter(spectrogram)

        features_audio = self.audio_encoder(augmented_audio, training=False)
        features_spectrogram = self.spectrogram_encoder(augmented_spectrogram, training=False)

        projections_audio = self.audio_projection_head(features_audio, training=False)
        projections_spectrogram = self.spectrogram_projection_head(features_spectrogram, training=False)
        
        contrastive_loss = contrastive_loss_fn(projections_audio, projections_spectrogram, self.tau)+contrastive_loss_fn(projections_spectrogram, projections_audio, self.tau)
        self.contrastive_loss_tracker.update_state(contrastive_loss)

        cosine_similarity = tf.reduce_mean(keras.losses.cosine_similarity(projections_audio, projections_spectrogram))
        self.mean_cosine_similarity.update_state(cosine_similarity)

        return {m.name: m.result() for m in self.metrics}

Compile and train MF-CLR model

In [11]:
MF_CLR_model = MF_CLRModel()

early_stopping = EarlyStopping(
    monitor='val_c_loss',  
    patience=3,       
    restore_best_weights=True, 
    mode='min' 
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_c_loss', 
    factor=0.1,         
    patience=2          
)

MF_CLR_model.compile(
    audio_optimizer=keras.optimizers.Adam(),
    spectrogram_optimizer=keras.optimizers.Adam(),
)

pretraining_history = MF_CLR_model.fit(
    ds_train.repeat(), 
    epochs=30, 
    validation_data=ds_val, 
    steps_per_epoch=int((0.7*ds_size) // batch_size),
    callbacks=[early_stopping, reduce_lr]
)

In [12]:
fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('Epochs')
ax1.set_ylabel('c_loss', color=color)
ax1.plot(pretraining_history.history['c_loss'], color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

color = 'tab:blue'
ax2.set_ylabel('mean_cosine_similarity', color=color) 
ax2.plot(pretraining_history.history['mean_cosine_similarity'], color=color)
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()
plt.title('Evolution of metrics')
plt.show()

In [13]:
MF_CLR_model.save_weights('mfclr.weights.h5')
MF_CLR_model.load_weights('mfclr.weights.h5')

Train the classification head

In [14]:
MF_CLR_model.spectrogram_encoder.trainable = False
MF_CLR_model.audio_encoder.trainable = False

In [17]:
tf.config.experimental_run_functions_eagerly(True)

class Classification_model_mix(keras.Model):
    def __init__(self):
        super().__init__()
        self.tau = tau
        self.spectrogram_encoder = MF_CLR_model.spectrogram_encoder
        self.audio_encoder = MF_CLR_model.audio_encoder
        self.classification_head=keras.Sequential([
            layers.Input(shape=(2*n_embedding,)),
            layers.Dense(2*128, activation='relu'),
            layers.BatchNormalization(),
            layers.Dense(30) #nb labels
        ], name='Classification_head')


    def compile(self, optimizer,**kwargs):
        super().compile(**kwargs)

        self.optimizer = optimizer
        self.loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.accuracy_tracker = keras.metrics.SparseCategoricalAccuracy(name="accuracy")

    @property
    def metrics(self):
        return [
            self.loss_tracker,
            self.accuracy_tracker,
        ]

    def train_step(self, data):
        ((audio, labels), (spectrogram, _)) = data

        with tf.GradientTape() as tape:
            features_audio = self.audio_encoder(audio)
            features_spectrogram = self.spectrogram_encoder(spectrogram)
            logits=self.classification_head(tf.concat([features_audio, features_spectrogram], axis=1))
            loss = self.loss_fn(labels, logits)

        gradients= tape.gradient(
            loss,
            self.classification_head.trainable_weights,
        )
        self.optimizer.apply_gradients(
            zip(gradients, self.classification_head.trainable_weights)
        )

        self.loss_tracker.update_state(loss)
        self.accuracy_tracker.update_state(labels, logits)

        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        (audio, labels), (spectrogram, _) = data

        features_audio = self.audio_encoder(audio, training=False)
        features_spectrogram = self.spectrogram_encoder(spectrogram, training=False)
        concatenated_features = tf.concat([features_audio, features_spectrogram], axis=1)
        logits = self.classification_head(concatenated_features, training=False)
        loss = self.loss_fn(labels, logits)
        self.loss_tracker.update_state(loss)
        self.accuracy_tracker.update_state(labels, logits)

        return {m.name: m.result() for m in self.metrics}

early_stopping = EarlyStopping(
    monitor='val_loss',  
    patience=4,       
    restore_best_weights=True  
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5,         
    patience=2          
)

classification_model_mix=Classification_model_mix()
classification_model_mix.compile(
    optimizer=keras.optimizers.Adam(1e-4), 
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
    metrics=['accuracy'])

classification_model_mix.fit(
    ds_train.repeat(), 
    epochs=60, 
    validation_data=ds_val, 
    steps_per_epoch=int((0.7*ds_size) // batch_size),
    callbacks=[early_stopping, reduce_lr])

Epoch 1/60


2024-07-10 09:12:07.883828: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:113: Filling up shuffle buffer (this may take a while): 618 of 1843
2024-07-10 09:12:17.886279: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:113: Filling up shuffle buffer (this may take a while): 1275 of 1843
2024-07-10 09:12:25.002170: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-10 09:12:25.035964: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-10 09:12:25.036639: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.
2024-07-10 09:12:25.047690: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.0295 - loss: 348.4432

2024-07-10 09:14:33.951344: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:113: Filling up shuffle buffer (this may take a while): 766 of 1843
2024-07-10 09:14:50.595448: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-10 09:14:50.624891: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.
2024-07-10 09:14:50.626147: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


40/40 ━━━━━━━━━━━━━━━━━━━━ 185s 4s/step - accuracy: 0.0294 - loss: 346.5448 - val_accuracy: 0.0653 - val_loss: 124.0983 - learning_rate: 1.0000e-04
Epoch 2/60


...

40/40 ━━━━━━━━━━━━━━━━━━━━ 242s 6s/step - accuracy: 0.8435 - loss: 0.8485 - val_accuracy: 0.8920 - val_loss: 0.7962 - learning_rate: 2.5000e-05
Epoch 21/60


2024-07-10 10:20:58.252602: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.8564 - loss: 0.7196

2024-07-10 10:23:48.986354: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:113: Filling up shuffle buffer (this may take a while): 594 of 1843
2024-07-10 10:24:07.076238: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-10 10:24:07.106317: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.
2024-07-10 10:24:20.618693: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-10 10:24:30.585494: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


40/40 ━━━━━━━━━━━━━━━━━━━━ 243s 6s/step - accuracy: 0.8565 - loss: 0.7201 - val_accuracy: 0.8750 - val_loss: 0.6016 - learning_rate: 2.5000e-05
Epoch 22/60
34/40 ━━━━━━━━━━━━━━━━━━━━ 25s 4s/step - accuracy: 0.8564 - loss: 0.6914

In [ ]:
evaluation = classification_model_mix.evaluate(ds_test)

# Print the evaluation metrics
print("Evaluation Mix Accuracy: {:.2f}%".format(evaluation[1]*100))

2024-07-09 18:15:25.321017: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:113: Filling up shuffle buffer (this may take a while): 987 of 1843
2024-07-09 18:15:33.751868: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2024-07-09 18:15:33.786338: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.
2024-07-09 18:15:33.786458: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


185/185 ━━━━━━━━━━━━━━━━━━━━ 37s 89ms/step - accuracy: 0.9691 - loss: 0.1853
Evaluation Mix Accuracy: 95.68%


2024-07-09 18:15:52.262380: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
